# accel-sim silicon anchor — attention profiler

Times **naive** (hand-written QK^T → softmax → V, materializing the full score tensor)
vs **fused** (PyTorch's `scaled_dot_product_attention`, a real flash/memory-efficient kernel)
multi-head self-attention on this GPU.

**Before running:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`.

Writes `attention_profile.json`, prints it, and auto-downloads it. Bring that file back and run:

```bash
python validate/silicon/compare_attention.py attention_profile.json
```


In [ ]:
# ---- config (edit if you want) ---------------------------------------------
# NOTE: on a Colab T4 (Turing) or V100 use DTYPE = "fp16" -- those GPUs have
# NO bf16 tensor cores and bf16 silently falls back to a ~20x-slower kernel.
DTYPE  = "fp16"          # "bf16" | "fp16" | "fp32"
BATCH  = 8
SEQ    = 1024
HEADS  = 12
D_HEAD = 64
ITERS  = 50
WARMUP = 15
OUT    = "attention_profile.json"


In [ ]:
_DTYPES = {"bf16": "bfloat16", "fp16": "float16", "fp32": "float32"}

import torch
import torch.nn.functional as F
assert torch.cuda.is_available(), "no CUDA device -- Runtime > Change runtime type > T4 GPU"

device = torch.device("cuda")
dtype = getattr(torch, _DTYPES[DTYPE])
B, S, H, Dh = BATCH, SEQ, HEADS, D_HEAD
gpu = torch.cuda.get_device_name(0)
print(f"GPU: {gpu}   dtype={DTYPE}   B={B} S={S} H={H} Dh={Dh}   iters={ITERS} (+{WARMUP} warmup)")


In [ ]:
import math, statistics

def bench(fused):
    q = torch.randn(B, H, S, Dh, device=device, dtype=dtype, requires_grad=True)
    k = torch.randn(B, H, S, Dh, device=device, dtype=dtype, requires_grad=True)
    v = torch.randn(B, H, S, Dh, device=device, dtype=dtype, requires_grad=True)
    scale = 1.0 / math.sqrt(Dh)

    def naive_forward():
        scores = (q @ k.transpose(-2, -1)) * scale
        return torch.softmax(scores, dim=-1) @ v

    def fused_forward():
        return F.scaled_dot_product_attention(q, k, v)

    forward_fn = fused_forward if fused else naive_forward
    fwd, bwd = [], []
    for i in range(WARMUP + ITERS):
        for t in (q, k, v):
            t.grad = None
        # 4 events, not 3: bracket the loss reduction into "forward" so it
        # isn't silently counted as part of "backward".
        ev = [torch.cuda.Event(enable_timing=True) for _ in range(4)]
        ev[0].record()
        out = forward_fn()
        ev[1].record()
        loss = out.float().square().mean()
        ev[2].record()
        loss.backward()
        ev[3].record()
        torch.cuda.synchronize()
        if i >= WARMUP:
            fwd.append(ev[0].elapsed_time(ev[2]))
            bwd.append(ev[2].elapsed_time(ev[3]))

    def stat(v):
        return {"mean_ms": statistics.fmean(v),
                "std_ms": statistics.pstdev(v) if len(v) > 1 else 0.0}
    return {"forward": stat(fwd), "backward": stat(bwd)}


In [ ]:
naive = bench(fused=False)
print(f"  naive  fwd {naive['forward']['mean_ms']:8.3f}  bwd {naive['backward']['mean_ms']:8.3f} ms")
fused = bench(fused=True)
print(f"  fused  fwd {fused['forward']['mean_ms']:8.3f}  bwd {fused['backward']['mean_ms']:8.3f} ms")
tot_naive = naive["forward"]["mean_ms"] + naive["backward"]["mean_ms"]
tot_fused = fused["forward"]["mean_ms"] + fused["backward"]["mean_ms"]
print(f"  measured fusion saving: {(tot_naive - tot_fused) / tot_naive * 100:.1f}%")


In [ ]:
# a quick kernel-name check that the fused path really dispatched to a fused
# CUDA kernel and didn't silently fall back to the generic math backend
fused_top_ops = []
try:
    from torch.profiler import profile, ProfilerActivity
    q = torch.randn(B, H, S, Dh, device=device, dtype=dtype, requires_grad=True)
    k = torch.randn(B, H, S, Dh, device=device, dtype=dtype, requires_grad=True)
    v = torch.randn(B, H, S, Dh, device=device, dtype=dtype, requires_grad=True)
    for _ in range(5):
        F.scaled_dot_product_attention(q, k, v).sum().backward()
        for t in (q, k, v):
            t.grad = None
    torch.cuda.synchronize()
    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
        F.scaled_dot_product_attention(q, k, v).sum().backward()
        torch.cuda.synchronize()
    for e in prof.key_averages():
        cuda_us = getattr(e, "self_device_time_total", 0) or getattr(e, "self_cuda_time_total", 0)
        if cuda_us > 0:
            fused_top_ops.append({"op": e.key, "cuda_ms": cuda_us / 1e3, "count": e.count})
    fused_top_ops.sort(key=lambda r: -r["cuda_ms"])
    fused_top_ops = fused_top_ops[:10]
    print("\ntop fused-path kernels (check for 'flash'/'efficient', not 'math'/'softmax'+'bmm' separately):")
    for r in fused_top_ops[:5]:
        print(f"  {r['cuda_ms']:7.3f} ms  x{r['count']:3d}  {r['op']}")
except Exception as e:
    print(f"(kernel trace skipped: {e})")


In [ ]:
import json, platform

out = {
    "gpu": gpu, "torch": torch.__version__, "cuda": torch.version.cuda,
    "dtype": _DTYPES[DTYPE], "batch": B, "seq": S, "n_heads": H, "d_head": Dh,
    "iters": ITERS, "warmup": WARMUP, "naive": naive, "fused": fused,
    "fused_top_ops": fused_top_ops, "host": platform.platform(),
}
with open(OUT, "w") as f:
    json.dump(out, f, indent=2)

print(f"\n===== {OUT} (copy this back if the download fails) =====\n")
print(json.dumps(out, indent=2))

try:
    from google.colab import files
    files.download(OUT)
except Exception as e:
    print(f"\n(auto-download unavailable: {e} -- grab {OUT} from the Files sidebar)")
